Model Cookbook
==============

The model cookbook provides a concise reference to model composition tools, specifically the `Model` and
`Collection` objects.

Examples using different PyAutoGalaxy API’s for model composition are provided, which produce more concise and
readable code for different use-cases.

__Contents__

- **Simple Model:** Compose a model with a galaxy having a single Sersic light profile.
- **More Complex Models:** Compose models with multiple light profiles and multiple galaxies.
- **Concise API:** A shorthand API for composing models more concisely.
- **Prior Customization:** Customize the priors of individual model parameters.
- **Model Customization:** Pair, fix and offset model parameters to reduce complexity.
- **Available Model Components:** Links to API documentation for all available light and mass profiles.
- **JSon Outputs:** Save and load models as JSON files.
- **Many Profile Models (Advanced):** Compose models with many profiles using MGE or shapelets.
- **Model Linking (Advanced):** Link inferred models between searches in a chain.
- **Across Datasets (Advanced):** Compose models that share components across multiple datasets.
- **Relations (Advanced):** Compose models where parameters vary according to a user-specified function.
- **PyAutoFit API:** Links to the PyAutoFit model composition cookbooks for more advanced usage.
- **Wrap Up:** Summary of model composition tools.

__Start Here Notebook__

If any code in this script is unclear, refer to the `imaging/modeling.ipynb` notebook.

__Google Colab Setup__

This cell sets up the environment when the notebook is run on Google Colab: it installs the
required PyAuto packages, clones the workspace (configuration files and example datasets) and
points the configuration at it. If you are running the notebook elsewhere (e.g. locally via
your own installation) it does nothing, and you can run it safely.

Colab tip: model-fits run much faster on a GPU — enable one via "Runtime" -> "Change runtime
type" -> "Hardware accelerator" before running the notebook.

In [ ]:
try:
    import google.colab
except ImportError:
    from autogalaxy import setup_colab as _setup_colab
else:
    import importlib
    import subprocess
    import sys

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "autonerves", "--no-deps"]
    )
    _setup_colab = importlib.import_module("autonerves.setup_colab")

_setup_colab.setup("autogalaxy")

In [ ]:

from autogalaxy import setup_notebook; setup_notebook()

from pathlib import Path
import autofit as af
import autogalaxy as ag

__Simple Model__

A simple model we can compose has a galaxy with a Sersic light profile:

In [ ]:

bulge = af.Model(ag.lp_linear.Sersic)

galaxy = af.Model(ag.Galaxy, redshift=0.5, bulge=bulge)

model = af.Collection(galaxies=af.Collection(galaxy=galaxy))

print(model.info)

The same model can also be drawn as a figure, which shows its structure at a glance.

The figure is the **map** and `model.info` is the **legend**. The map shows the shape of the model: which component 
owns which parameter, and what state every parameter is in (free, fixed, shared with another component, related to 
one by an expression, solved during the fit or missing from your configuration). The legend gives the numbers: the 
prior on every parameter and the value of every fixed one. The dashed `intensity · solved` pill in the figure is the 
parameter `model.info` does not print, because it is not part of the model: the `intensity` of a linear light profile 
is solved for by the inversion at every likelihood evaluation.

In [ ]:
af.ModelPlotter(model).figure()

The model `total_free_parameters` tells us the total number of free parameters (which are fitted for via a 
non-linear search), which in this case is 6. The `intensity` of a linear light profile is not one of them: it is 
solved for by a linear inversion during every likelihood evaluation.

In [ ]:
print(f"Model Total Free Parameters = {model.total_free_parameters}")

If we print the `info` attribute of the model we get information on all of the parameters and their priors.

This is the same model as above, so it draws the same figure; nothing new to see.

In [ ]:
print(model.info)

__More Complex Models__

The API above can be easily extended to compose models where each galaxy has multiple light or mass profiles:

In [ ]:
bulge = af.Model(ag.lp_linear.Sersic)
disk = af.Model(ag.lp_linear.Exponential)
bar = af.Model(ag.lp_linear.Sersic)

galaxy = af.Model(ag.Galaxy, redshift=0.5, bulge=bulge, disk=disk, bar=bar)

model = af.Collection(galaxies=af.Collection(galaxy=galaxy))

print(model.info)

The `bulge` and the `bar` are both `Sersic` profiles, so the figure collapses them into a single dashed plate badged 
`2 components` rather than drawing two identical cards, while the `Exponential` disk keeps its own card. Each of the 
three linear profiles carries its own dashed `intensity · solved` pill, which the footer counts as 3 parameters 
solved during fitting.

In [ ]:
af.ModelPlotter(model).figure()

The use of the words `bulge`, `disk` and `bar` above are arbitrary. They can be replaced with any name you
like, e.g. `bulge_0`, `bulge_1`, `star_clump`, and the model will still behave in the same way.

The API can also be extended to compose models where there are multiple galaxies:

In [ ]:
bulge = af.Model(ag.lp_linear.Sersic)

galaxy_0 = af.Model(
    ag.Galaxy,
    redshift=0.5,
    bulge=bulge,
)

bulge = af.Model(ag.lp_linear.Sersic)

galaxy_1 = af.Model(
    ag.Galaxy,
    redshift=0.5,
    bulge=bulge,
)

model = af.Collection(
    galaxies=af.Collection(
        galaxy_0=galaxy_0,
        galaxy_1=galaxy_1,
    )
)

print(model.info)

The two galaxies are identical in structure, so the figure draws them once inside a dashed plate badged 
`2 components` rather than twice. Their parameters are badged `independent`: two separate priors with the same 
configuration, which is not the same thing as one shared prior.

In [ ]:
af.ModelPlotter(model).figure()

__Concise API__

If a light profile is passed directly to the `af.Model` of a galaxy, it is automatically assigned to be a `af.Model` 
component of the galaxy.

This means we can write the model above comprising multiple light profiles more concisely as follows:

In [ ]:
galaxy = af.Model(
    ag.Galaxy,
    redshift=0.5,
    bulge=ag.lp_linear.Sersic,
    disk=ag.lp_linear.Exponential,
    bar=ag.lp_linear.Sersic,
)

model = af.Collection(galaxies=af.Collection(galaxy=galaxy))

print(model.info)

The figure is identical to the one drawn the long way round, which is the point: the concise API is a shorthand for 
writing the model, not a different model.

In [ ]:
af.ModelPlotter(model).figure()

__Prior Customization__

We can customize the priors of the model component individual parameters as follows:

In [ ]:
bulge = af.Model(ag.lp_linear.Sersic)
bulge.centre.centre_0 = af.UniformPrior(lower_limit=-0.1, upper_limit=0.1)
bulge.centre.centre_1 = af.UniformPrior(lower_limit=-0.1, upper_limit=0.1)
bulge.sersic_index = af.TruncatedGaussianPrior(
    mean=4.0, sigma=1.0, lower_limit=1.0, upper_limit=8.0
)

galaxy = af.Model(
    ag.Galaxy,
    redshift=0.5,
    bulge=bulge,
)

model = af.Collection(galaxies=af.Collection(galaxy=galaxy))

print(model.info)

Customizing a prior does not change a parameter's state: every parameter above is still sampled, so the figure is 
unchanged by the customization and is the same map as the simple model at the top of this cookbook. Print 
`model.info`, or call `af.ModelPlotter(model).figure(detail="priors")`, to see the prior on each parameter.

In [ ]:
af.ModelPlotter(model).figure()

__Model Customization__

We can customize the model parameters in a number of different ways, as shown below:

In [ ]:
bulge = af.Model(ag.lp_linear.Sersic)
disk = af.Model(ag.lp_linear.Exponential)

# Parameter Pairing: Pair the centre of the bulge and disk together, reducing
# the complexity of non-linear parameter space by N = 2

bulge.centre = disk.centre

# Parameter Fixing: Fix the sersic_index of the bulge to a value of 4, reducing
# the complexity of non-linear parameter space by N = 1

bulge.sersic_index = 4.0

# Parameter Offsets: Make the bulge effective_radius parameters the same value as
# the disk but with an offset.

bulge.effective_radius = disk.effective_radius + 0.1

galaxy = af.Model(
    ag.Galaxy,
    redshift=0.5,
    bulge=bulge,
    disk=disk,
)

model = af.Collection(galaxies=af.Collection(galaxy=galaxy))

# Assert that the effective radius of the bulge is larger than that of the disk.
# (Assertions can only be added at the end of model composition, after all components
# have been bright together in a `Collection`.
model.add_assertion(
    model.galaxies.galaxy.bulge.effective_radius
    > model.galaxies.galaxy.disk.effective_radius
)

# Assert that the bulge effetive radius is below 3.0":
model.add_assertion(model.galaxies.galaxy.bulge.effective_radius < 3.0)

print(model.info)

This is the stage where the figure earns its keep: the paired `centre` is drawn once on the `bulge` badged 
`shared ×2`, with a blue link from the `disk` that reuses it (`↗ bulge.centre`), the fixed `sersic_index` is a grey 
pill, the offset `effective_radius` carries its defining expression on the pill, and each assertion is a compact 
dashed-orange label naming both of its operands.

In [ ]:
af.ModelPlotter(model).figure()

__Available Model Components__

The light profiles, mass profiles and other components that can be used for galaxy modeling are given at the following
API documentation pages:

 - https://pyautogalaxy.readthedocs.io/en/latest/api/light.html
 - https://pyautogalaxy.readthedocs.io/en/latest/api/mass.html
 - https://pyautogalaxy.readthedocs.io/en/latest/api/pixelization.html
 
 __JSon Outputs__
 
 After a model is composed, it can easily be output to a .json file on hard-disk in a readable structure:

In [ ]:
import os
import json

model_path = Path("path", "to", "model", "json")

os.makedirs(model_path, exist_ok=True)

model_file = Path(model_path, "model.json")

with open(model_file, "w+") as f:
    json.dump(model.dict(), f, indent=4)

We can load the model from its `.json` file.

In [ ]:
model = af.Model.from_json(file=model_file)

print(model.info)

The reloaded model is the same model, so `af.ModelPlotter(model).figure()` draws the identical figure.

This means in **PyAutoGalaxy** one can write a model in a script, save it to hard disk and load it elsewhere, as well
as manually customize it in the .json file directory.

__Many Profile Models (Advanced)__

Features such as the Multi Gaussian Expansion (MGE) and shapelets compose models consisting of 50 - 500+ light
profiles.

The following example notebooks show how to compose and fit these models:

https://github.com/PyAutoLabs/autogalaxy_workspace/blob/main/notebooks/imaging/features/multi_gaussian_expansion/modeling.ipynb
https://github.com/PyAutoLabs/autogalaxy_workspace/blob/main/notebooks/imaging/features/shapelets/modeling.ipynb

__Model Linking (Advanced)__

When performing non-linear search chaining, the inferred model of one phase can be linked to the model.

The following example notebooks show how to compose and fit these models:

https://github.com/PyAutoLabs/autogalaxy_workspace/blob/main/notebooks/imaging/advanced/guides/modeling/chaining.ipynb

__Across Datasets (Advanced)__

When fitting multiple datasets, model can be composed where the same model component are used across the datasets
but certain parameters are free to vary across the datasets.

The following example notebooks show how to compose and fit these models:

https://github.com/PyAutoLabs/autogalaxy_workspace/blob/main/notebooks/multi_dataset/start_here.ipynb

__Relations (Advanced)__

We can compose models where the free parameter(s) vary according to a user-specified function 
(e.g. y = mx +c -> effective_radius = (m * wavelength) + c across the datasets.

The following example notebooks show how to compose and fit these models:

https://github.com/PyAutoLabs/autogalaxy_workspace/blob/main/notebooks/multi_dataset/features/wavelength_dependence/modeling.ipynb

__PyAutoFit API__

**PyAutoFit** is a general model composition library which offers even more ways to compose models not
detailed in this cookbook.

The **PyAutoFit** model composition cookbooks detail this API in more detail:

https://pyautofit.readthedocs.io/en/latest/cookbooks/model.html
https://pyautofit.readthedocs.io/en/latest/cookbooks/multi_level_model.html

__Wrap Up__

This cookbook shows how to compose simple models using the `af.Model()` and `af.Collection()` objects.